In [1]:
import os
import pandas as pd
import torch
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

C:\Users\USER\PycharmProjects\DSGP15_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load dataset from local path

In [2]:
ROOT_DIR = r"C:\Users\USER\PycharmProjects\DSGP15_Project\ml-models\dataset\Dataset"

EMOTION_TRAIN_CSV_PATH = os.path.join(ROOT_DIR, "Texts", "Emotion", "Emotion_Train.csv")
EMOTION_TEST_CSV_PATH  = os.path.join(ROOT_DIR, "Texts", "Emotion", "Emotion_Test.csv")

emo_train = pd.read_csv(EMOTION_TRAIN_CSV_PATH, header=None)
emo_train.columns = ['ImageID', 'Desc_Turk', 'Desc_Eng', 'Emotion', 'Invalid']

# Fix incorrectly placed English descriptions
mask = emo_train['Invalid'].notnull()
emo_train.loc[mask, 'Desc_Eng'] = emo_train.loc[mask, 'Invalid']
emo_train = emo_train.drop(columns=['Invalid'])

emo_test = pd.read_csv(EMOTION_TEST_CSV_PATH, header=None)
emo_test.columns = ['ImageID', 'Desc_Turk', 'Desc_Eng', 'Emotion']

Prepare training data

In [3]:
train_df = emo_train[['Desc_Eng', 'Emotion']].dropna()
test_df  = emo_test[['Desc_Eng', 'Emotion']].dropna()

# Encode emotion labels
le = LabelEncoder()
train_df['label'] = le.fit_transform(train_df['Emotion'])
test_df['label'] = le.transform(test_df['Emotion'])

X_train, X_val, y_train, y_val = train_test_split(
    train_df['Desc_Eng'],
    train_df['label'],
    test_size=0.2,
    random_state=42
)

## Tokenize text using BERT

In [4]:
model_name = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(model_name)

train_encodings = tokenizer(list(X_train), truncation=True, padding=True, max_length=128)
val_encodings   = tokenizer(list(X_val), truncation=True, padding=True, max_length=128)


Create PyTorch dataset class

In [5]:
class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmotionDataset(train_encodings, list(y_train))
val_dataset   = EmotionDataset(val_encodings, list(y_val))

Build BERT model

In [6]:
model = BertForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_)
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Training arguments

In [7]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,    # CPU-friendly
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    logging_steps=50,
    save_steps=500,
    save_total_limit=1,
    report_to="none",
    no_cuda=True                      # Force CPU usage
)

Train the model

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    logging_steps=50,
    save_steps=500
)


Prediction function

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

def predict_emotion(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    pred_id = torch.argmax(logits, dim=1).item()
    return le.inverse_transform([pred_id])[0]


GET PREDICTIONS FROM THE TRAINED MODEL

In [12]:
val_outputs = trainer.predict(val_dataset)

y_val_true = y_val.values
y_val_pred = np.argmax(val_outputs.predictions, axis=1)
y_val_probs = torch.softmax(torch.tensor(val_outputs.predictions), dim=1).numpy()


AttributeError: `AcceleratorState` object has no attribute `distributed_type`. This happens if `AcceleratorState._reset_state()` was called and an `Accelerator` or `PartialState` was not reinitialized.

CLASSIFICATION REPORT

In [ ]:
print("Classification Report:\n")
print(classification_report(
    y_val_true,
    y_val_pred,
    target_names=le.classes_
))


CONFUSION MATRIX

In [ ]:
cm = confusion_matrix(y_val_true, y_val_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=le.classes_,
    yticklabels=le.classes_
)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()


ROC CURVE

In [ ]:
plt.figure(figsize=(7, 6))

if len(le.classes_) == 2:
    # Binary classification
    fpr, tpr, _ = roc_curve(y_val_true, y_val_probs[:, 1])
    roc_auc = auc(fpr, tpr)

    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (Binary)")
    plt.legend()

else:
    # Multiclass ROC
    for i, label in enumerate(le.classes_):
        fpr, tpr, _ = roc_curve(y_val_true == i, y_val_probs[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.2f})")

    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (Multiclass)")
    plt.legend()

plt.show()


In [ ]:
print(predict_emotion("The child looks very happy and excited"))
print(predict_emotion("The drawing shows sadness and fear"))
print(predict_emotion("This picture feels angry and dark"))
